# HighPitchCT — Stage 6 (`v0.9.0`): التعميم خارج العينة بدون تسرب بيانات (`Zero-Leakage Generalization`)

يختبر هذا الدفتر تعميم الشبكة ثنائية المجال ذات المستويات الأربعة (`64 -> 128 -> 256 -> 512`) والتدريب المتسلسل ذي الـ 3 مراحل (`Eq. 1 -> Eq. 2 -> Eq. 3`) على شرائح/مرضى لم ترهم الشبكة أبداً أثناء التدريب (`Out-of-Sample Zero-Shot Evaluation`).

In [ ]:
import os
import sys
import subprocess
import json
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

REPO_DIR = Path('/kaggle/working/HighPitchCT-Pipeline')
DATA_ROOT = Path('/kaggle/working/highpitchct-data')

subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', 'main'], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'reset', '--hard', 'origin/main'], check=True)

env = os.environ.copy()
env['PYTHONPATH'] = str(REPO_DIR / 'src')

cmd = [
    sys.executable, '-m', 'highpitchct.cli', 'stage6-generalize',
    '--data-root', str(DATA_ROOT),
    '--case-id', 'C012',
    '--split-mode', 'axial_guardband',
    '--train-slices-per-case', '2',
    '--test-slices-per-case', '1',
    '--min-guardband-mm', '45.0',
    '--backend', 'torch_cuda',
    '--pitches', '1.5,2.0,2.5',
    '--itf-ratio', '0.60',
    '--sino-channels', '64',
    '--img-channels', '64',
    '--epochs', '36',
    '--lr', '1.5e-3',
    '--save-model',
    '--no-progress',
]

res = subprocess.run(cmd, env=env, capture_output=True, text=True)
if res.returncode != 0:
    print('STDERR:\n', res.stderr)
    raise RuntimeError(f'stage6-generalize failed with code {res.returncode}')

report = json.loads(res.stdout)
real_gen = report['real_patient_generalization']
print('الإصدار:', report['stage'])
print('تدقيق عدم تسرب البيانات (Zero-Leakage Audit):', json.dumps(real_gen['zero_leakage_audit'], indent=2, ensure_ascii=False))
print('مقاييس التعميم على بيانات الاختبار غير المرئية:', json.dumps(real_gen['generalization_metrics'], indent=2, ensure_ascii=False))

rows = []
for s in real_gen['unseen_test_summary']['per_sample']:
    rows.append({
        'عينة الاختبار غير المرئية (Unseen Test Sample)': s['sample_id'],
        'السيناريو': s['scenario'].upper(),
        'Pitch (Target / Eff / Native)': f"{s['target_pitch']:.1f} / {s['effective_pitch']:.2f} / {s['native_pitch']:.2f}",
        'نسبة الفجوة المفقودة': f"{s['missing_fraction']*100:.1f}%",
        'RMSE Raw No-Fill [HU]': f"{s['raw_nofill_rmse_hu']:.2f}",
        'RMSE Prelim FBP [HU]': f"{s['prelim_rmse_hu']:.2f}",
        'RMSE Domain-1 (Zero-Shot) [HU]': f"{s['domain1_dc_rmse_hu']:.2f}",
        'RMSE Final Dual-Domain (Zero-Shot) [HU]': f"{s['final_dual_rmse_hu']:.2f}",
        'SSIM (Prelim -> Dual)': f"{s['prelim_ssim']:.4f} -> {s['final_dual_ssim']:.4f}",
        'Wedge MAE (Reproj -> SinoNet)': f"{s['reproj_missing_wedge_mae']:.4f} -> {s['sinonet_dc_missing_wedge_mae']:.4f}",
    })

display(pd.DataFrame(rows))
display(Image(filename=real_gen['visual_qa_png']))
